# FitBrand Co. — Data Pipeline & Synthetic ETL
Simulation of D2C transactions with category-based depletion cycles and channel-specific retention probabilities.

In [1]:
!pip install faker

In [2]:
import random
from datetime import datetime, timedelta
import pandas as pd
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

CLIENTS_NUM = 1000

PRODUCTS = {
    "Whey Protein": {"price": 55, "cycle_days": 30},
    "Pre-Workout": {"price": 40, "cycle_days": 45},
    "Creatine": {"price": 35, "cycle_days": 60},
    "Multivitamin": {"price": 25, "cycle_days": 90},
}

CHANNELS = ["TikTok", "Instagram", "Ambassador", "Organic"]

data_transactions = []

for client_id in range(1001, 1001 + CLIENTS_NUM):
    channel = random.choice(CHANNELS)
    age = random.randint(18, 45)

    # Retention based on acquisition channel trust
    retention_prob = (
        0.65 if channel in ["Ambassador", "Organic"]
        else (0.40 if channel == "Instagram" else 0.25)
    )

    initial_date = fake.date_between(start_date="-1y", end_date="-6m")
    prod_name = random.choice(list(PRODUCTS.keys()))
    prod_info = PRODUCTS[prod_name]

    # Initial Acquisition Transaction
    data_transactions.append({
        "client_id": client_id,
        "age": age,
        "acquisition_channel": channel,
        "product": prod_name,
        "purchase_amount_usd": prod_info["price"],
        "purchase_date": initial_date,
        "purchase_number": 1,
    })

    # Repurchase Cycle
    if random.random() < retention_prob:
        num_repurchases = random.randint(1, 4)
        current_date = initial_date

        for i in range(2, num_repurchases + 2):
            variability_days = random.randint(-5, 15)
            next_purchase_days = prod_info["cycle_days"] + variability_days
            current_date = current_date + timedelta(days=next_purchase_days)

            if current_date > datetime.now().date():
                break

            # ¡El bloque append ahora está alineado correctamente con el if, fuera del break!
            data_transactions.append({
                "client_id": client_id,
                "age": age,
                "acquisition_channel": channel,
                "product": prod_name,
                "purchase_amount_usd": prod_info["price"],
                "purchase_date": current_date,
                "purchase_number": i,
            })

df = pd.DataFrame(data_transactions)
df.to_csv("fitbrand_transactions.csv", index=False)

# Data Sanitation
print(f"Dataset generated: {len(df)} transactions across {CLIENTS_NUM} unique clients.")
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

Dataset generated: 1807 transactions across 1000 unique clients.
Missing values: 0
Duplicates: 0


## 2. SQLite Relational Data Ingestion
Loading the Raw Data layer into a local relational database (`fitbrand.db`) for analytical querying.

In [3]:
import sqlite3

conn = sqlite3.connect("fitbrand.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)

def run_query(query):
    return pd.read_sql_query(query, conn)

# Verification Query
run_query("SELECT * FROM transactions LIMIT 5;")

,client_id,age,acquisition_channel,product,purchase_amount_usd,purchase_date,purchase_number
0,1001,18,TikTok,Creatine,35,2026-05-14,1
1,1001,18,TikTok,Creatine,35,2026-07-11,2
2,1001,18,TikTok,Creatine,35,2026-09-21,3
3,1002,36,TikTok,Multivitamin,25,2025-10-02,1
4,1002,36,TikTok,Multivitamin,25,2026-01-01,2


## 3. SQL Analytical Queries
Calculating cumulative retention, channel LTV, and repurchase velocity using Window Functions.

In [4]:
query_retention = """
SELECT
    acquisition_channel,
    COUNT(DISTINCT client_id) AS total_clients,
    COUNT(DISTINCT CASE WHEN purchase_number > 1 THEN client_id END) AS retained_clients,
    ROUND(
        CAST(COUNT(DISTINCT CASE WHEN purchase_number > 1 THEN client_id END) AS FLOAT) /
        COUNT(DISTINCT client_id) * 100, 2
    ) AS retention_rate_pct,
    ROUND(CAST(SUM(purchase_amount_usd) AS FLOAT) / COUNT(DISTINCT client_id), 2) AS avg_ltv_usd
FROM transactions
GROUP BY acquisition_channel
ORDER BY avg_ltv_usd DESC;
"""
run_query(query_retention)

,acquisition_channel,total_clients,retained_clients,retention_rate_pct,avg_ltv_usd
0,Ambassador,259,142,54.83,84.29
1,Organic,238,126,52.94,83.38
2,Instagram,269,82,30.48,62.42
3,TikTok,234,51,21.79,54.32


In [5]:
query_repurchase_velocity = """
WITH ordered_purchases AS (
    SELECT
        client_id,
        product,
        purchase_date,
        LAG(purchase_date) OVER (
            PARTITION BY client_id
            ORDER BY purchase_number
        ) AS previous_purchase_date
    FROM transactions
)
SELECT
    product,
    ROUND(AVG(JULIANDAY(purchase_date) - JULIANDAY(previous_purchase_date)), 0) AS avg_days_to_repurchase
FROM ordered_purchases
WHERE previous_purchase_date IS NOT NULL
GROUP BY product;
"""

run_query(query_repurchase_velocity)

,product,avg_days_to_repurchase
0,Creatine,65.0
1,Multivitamin,95.0
2,Pre-Workout,51.0
3,Whey Protein,35.0


## 4. Retained LTV Analysis
Isolating customers with repeat purchases (`purchase_number > 1`) to evaluate the actual LTV of loyal users and filter out single-transaction churn noise across acquisition channels.

In [6]:
client_sales = df.groupby(['acquisition_channel', 'client_id'])['purchase_amount_usd'].sum().reset_index()
purchases = df.groupby('client_id')['purchase_number'].max().reset_index()
retained_clients = client_sales.merge(purchases, on='client_id').query('purchase_number > 1')

print(retained_clients.groupby('acquisition_channel')['purchase_amount_usd'].mean())

acquisition_channel
Ambassador    123.908451
Instagram     116.463415
Organic       125.000000
TikTok        111.078431
Name: purchase_amount_usd, dtype: float64
